# Where noise must enter to change the model's algorithm

**Demo of an interpretability experiment on Qwen3-8B. Question: where in the network does random noise have to
be injected before the model writes a *different correct algorithm*?**

Random (isotropic) noise added to Qwen3-8B's residual stream can make it write a different correct algorithm, for
example quickselect instead of a full sort, or DFS instead of BFS. This experiment asks **where** that noise has to
enter. One random direction per sampling slot is written into one of six *sites*, always at the same per-position
dose:

| Site | Where the noise goes |
|---|---|
| **S0** | task-specification tokens, blocks 12-23 |
| **S1** | read position R (last prompt token), blocks 12-23 |
| **S2** | condition sentence, blocks 12-23 |
| **S3** | generated tokens 1-32, blocks 12-23 |
| **S4** | task-specification tokens, blocks 0-11 |
| **S5** | task-specification tokens, blocks 24-35 |
| S1E / S2E | S1 / S2 with energy-matched doses |

**Full run (not repeated here).**

- **Model.** Qwen3-8B, revision `b968826d`, thinking off, temperature 1.0, top-p 1. It ran in FP8 W8A8 on a
  16 GB RTX 2000 Ada.
- **Tasks.** Six tasks, each with a *default* algorithm:

  | Task | Problem | Default |
  |---|---|---|
  | T03 | k-th smallest | full sort |
  | T05 | two sum | hash map |
  | T06 | find duplicate | hash set |
  | T10 | connected components | union-find |
  | T1 | grid path | BFS |
  | T2 | weighted shortest path | Dijkstra |

- **Slots.** A common-random-numbers (CRN) slot is one (task, template, k). It owns one random unit direction in
  the 4,096-d residual stream and one stream of token uniforms. Every arm on the slot shares both, so two arms on
  the same slot differ only in *where* the noise went. That makes every site contrast an exact paired flip.
- **Labels.** The run produced 4,510 programs. Each was labelled with the algorithm it implements (runtime and AST
  labellers, with an LLM judge on disagreement) and with pass/fail on tests.
- **Pricing.** Each site is priced on a *randomisation hull*. H5 is the best correct-only diversity (rarefied
  Hill-1) that can be reached within 0.05 pass-rate cost of the twin.
- **Published result.** Only noise in the task-specification states (S0, S4) changes the algorithm. S1 and S2 never
  do (0/526 and 0/527 correct off-default programs) and S5 almost never does (6/521). The pre-registered C1-site
  rule returns **PARTIAL**: S0 has the best H5 on 3 of the 6 tasks.

**What this notebook runs** (CPU only, under 10 minutes):
1. **Noise-injection maths.** The per-slot direction, the CRN inverse-CDF sampler and the displacement-targeted
   hook, on synthetic tensors (`gpu_main.py`, `engine.py`, `hooks.py`), plus the CPU unit tests
   (`tests/test_cpu.py`).
2. **The original analysis code** (`hull.py`, `analysis.py`): cells, hull prices H5/P15/P20 with a
   template-stratified bootstrap, the frozen C1-site decision, failure modes, paired flips with exact McNemar tests,
   the transmission GLM and the pooled mixed model. It runs on a curated subset: 90 of the 180 slots, covering all 6
   tasks and 5 templates with k = 0-2. Each cell therefore has N = 15 instead of 30.
3. **Figures** (`figures.py`), with the demo numbers beside the published full-run numbers.

**What it does not run.** Sampling Qwen3-8B (FP8 GPU engine, 1.74 GPU-hours), the LLM-judge labelling and the
novelty audit (OpenRouter). Their output, the labelled programs, is the data loaded below.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install
_pip('loguru==0.7.3')

# numpy, pandas, scipy, matplotlib, statsmodels, torch — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0', 'statsmodels==0.14.6')
    _pip('torch==2.9.0', '--index-url', 'https://download.pytorch.org/whl/cpu')

In [ ]:
from __future__ import annotations

# --- original import blocks (src/cfg.py, src/hooks.py, src/hull.py, src/analysis.py, tests/test_cpu.py, src/figures.py)
import hashlib
import json
import math
import os
import resource
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

from loguru import logger

from typing import Callable

import torch

import itertools

import numpy as np

import ast
import glob
import multiprocessing as mp
import time
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor

import matplotlib
import matplotlib.pyplot as plt  # figures.py called matplotlib.use("Agg"); dropped so figures render inline

# --- notebook additions
import http.client
import urllib.request
import warnings
import pandas as pd

warnings.formatwarning = lambda message, category, *args, **kwargs: f"{category.__name__}: {message}\n"  # no file paths

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-295f8e-prompt-bans-substitute-not-just-subtract/fork/run_jSassd-YWLVC/round-4/experiment-20/demo/mini_demo_data.json"
import http.client
import json
import urllib.request
from pathlib import Path

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=15) as response:
            return json.loads(response.read().decode())
    # Network errors, timeouts, a placeholder URL, bad JSON, a cut-off response.
    except (OSError, ValueError, http.client.HTTPException) as e:
        print("GitHub data unavailable, using local mini_demo_data.json:", e)
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
examples = data["datasets"][0]["examples"]
REF = data["metadata"]["full_run_reference"]  # published full-run numbers, for side-by-side comparison
print(data["metadata"]["description"])
print(f"{len(examples)} CRN slots, {sum(len(e['metadata_rows']) for e in examples)} labelled programs")
ex0 = examples[0]
print("\nexample slot:", ex0["input"][:600], "...")
print("\ntwin:", ex0["output"], "| S0 alpha 1/2/4:", ex0["predict_S0_a1"], ex0["predict_S0_a2"], ex0["predict_S0_a4"],
      "| S1 alpha 1/2/4:", ex0["predict_S1_a1"], ex0["predict_S1_a2"], ex0["predict_S1_a4"])

## Configuration

These are all the tunable knobs. The demo values below fit the 10-minute CPU budget, and each comment gives the
full-run value. The hull point estimates use the original settings. Only the number of bootstrap replicates is
reduced: the original used 2,000, at about 1 s each on this subset.

In [ ]:
DEMO_K = [0, 1, 2]    # CRN slots k used per (task, template): the demo data holds k 0-2. Original grid: k 0-5 (N = 30 per cell)
POINT_N_DRAWS = 1000  # CRN draws per hull point estimate (analysis.hull_all). Original: 1000
POINT_STEP = 0.05     # grid step of the hull mixture weights (point estimates). Original: 0.05
BOOT_B = 200          # template-stratified bootstrap replicates of every hull. Original: 2000 (`method.py analysis --boot 2000`)
BOOT_N_DRAWS = 200    # CRN draws per bootstrap hull (analysis._boot_task). Original: 200
BOOT_STEP = 0.1       # grid step of the mixture weights inside the bootstrap. Original: 0.1
WORKERS = 1           # the original spread the bootstrap over 5 spawn workers; the notebook runs it in-process

## 1. Frozen constants (`src/cfg.py`)

The tasks, their default algorithms, the sites and their blocks, the doses and the CRN slots were all frozen in
pre-registration commit #0. Two block conventions matter:

- **Block numbering.** Block `l` is decoder block `l`, and its output is HF `hidden_states[l + 1]`.
- **Positions.** S is the task-specification span, R = P - 1 is the read position, and COND is the condition
  sentence.

The path, logging-file and hardware helpers of `cfg.py` are left out because the notebook reads nothing from disk.

In [ ]:
# ------------------------------------------------------------------------------------------------ model
MODEL_ID = "Qwen/Qwen3-8B"
MODEL_REVISION = "b968826d9c46dd6066d109eabc6255188de91218"
N_LAYERS = 36
D_MODEL = 4096
IM_END = 151645
ENDOFTEXT = 151643
MAX_NEW_TOKENS = 1024
TEMPERATURE = 1.0
DECODING = "T1.0_topp1.0_topk-off_max1024_thinkoff_fencestop"
TEMPLATE_TAIL = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
QUANT = "fp8_e4m3_w8a8_dynamic(per-output-channel weight scales, per-token activation scales; lm_head/embeddings/norms bf16)"
BATCH = 20  # fixed batch shape (exp17's U1b-verified shape on this GPU); never changes within or across tasks

# ------------------------------------------------------------------------------------------------ tasks
TASKS = ("T03", "T05", "T06", "T10", "T1", "T2")
PATH_TASKS = ("T1", "T2")
BENCH_TASKS = ("T03", "T05", "T06", "T10")
DEFAULT = {"T1": "bfs", "T2": "dijkstra", "T03": "full_sort", "T05": "hash_map", "T06": "hash_set",
           "T10": "union_find"}
EXP8_TASK = {"T1": "t1", "T2": "t2"}
TEMPLATE_IDS = ["W1", "W2", "W3", "W4", "W5"]
TASK_ORDER_DROP = ("T10", "T06", "T05")  # fallback: whole tasks dropped in this order

# ------------------------------------------------------------------------------------------------ sites
PLAN_BLOCKS = list(range(12, 24))
LOW_BLOCKS = list(range(0, 12))
HIGH_BLOCKS = list(range(24, 36))
SITES = ("S0", "S1", "S2", "S3", "S4", "S5")
SITE_BLOCKS = {"S0": PLAN_BLOCKS, "S1": PLAN_BLOCKS, "S2": PLAN_BLOCKS, "S3": PLAN_BLOCKS, "S4": LOW_BLOCKS,
               "S5": HIGH_BLOCKS, "S1E": PLAN_BLOCKS, "S2E": PLAN_BLOCKS, "TOP5": PLAN_BLOCKS, "ENG": PLAN_BLOCKS}
SITE_POS = {"S0": "S", "S1": "R", "S2": "COND", "S3": "GEN1-32", "S4": "S", "S5": "S", "S1E": "R", "S2E": "COND",
            "TOP5": "S", "ENG": "S"}
S3_STEPS = 32  # generated input positions 1-32
NORM_BLOCKS = list(range(12, N_LAYERS))  # realised displacement at R logged at blocks 12-35
ALPHAS_GRID = (1.0, 2.0, 4.0)
ALPHA_BOUNDARY = 8.0
N_GRID_K = list(range(6))      # k 0-5 x 5 templates = N 30
N50_K = list(range(10))        # k 0-9 x 5 templates = N 50
SMOKE_K0 = 100                 # smoke slots k >= 100 are never analysed
FILL_K0 = 10                   # filler twin rows (TWIN_X) use k >= 10 (descriptive only)


# ------------------------------------------------------------------------------------------------ utils
def setup_logging(name: str) -> None:
    logger.remove()
    logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
    # notebook: the original also added a rotating file sink at logs/<name>.log


def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")


def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()


def stable_seed(*parts: Any) -> int:
    """Deterministic 63-bit seed from arbitrary parts (sha256 of their '|'-joined str) -- exp8/exp17 convention."""
    return int(hashlib.sha256("|".join(str(p) for p in parts).encode()).hexdigest(), 16) % (2 ** 63)


def wilson_ci(k: int, n: int, z: float = 1.959963984540054) -> tuple[float, float]:
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, c - h), min(1.0, c + h))

## 2. How the noise is injected (the GPU method, reduced to its maths)

The full engine (`src/engine.py`, `src/gpu_main.py`) needs Qwen3-8B on a GPU with FP8 support. The pieces that
define the *experimental design* are small, though, and they run here unchanged.

**2a. One CRN slot = one random direction + one stream of token uniforms.**

- **The direction.** `pn_dir(t, w, k)` seeds PCG64 with `stable_seed('PN4', t, w, k)` and normalises a 4,096-d
  Gaussian draw. Every site, dose and boundary arm of that slot uses this same `u`.
- **The uniforms.** `u_stream(t, w, k)` is the stream of token uniforms.
- **The sampler.** The inverse-CDF sampler `_inverse_cdf` turns each uniform into a token. When two arms on a slot
  have the same next-token distribution, they get the same token. That is why an arm whose noise does nothing
  reproduces the twin byte for byte.

In [ ]:
# src/gpu_main.py -- STEP 3 directions
def pn_dir(t: str, w: str, k: int) -> np.ndarray:
    x = np.random.Generator(np.random.PCG64(stable_seed("PN4", t, w, k))).standard_normal(D_MODEL)
    return (x / np.linalg.norm(x)).astype(np.float32)


# src/engine.py -- sampling
def u_stream(*key: Any, n: int = MAX_NEW_TOKENS) -> np.ndarray:
    """exp17/exp8 token uniforms: PCG64(stable_seed(*key)).random(n)."""
    return np.random.Generator(np.random.PCG64(stable_seed(*key))).random(n)


def _inverse_cdf(probs: torch.Tensor, u: torch.Tensor) -> torch.Tensor:
    cdf = probs.cumsum(dim=-1)
    cdf = cdf / cdf[:, -1:]
    tok = torch.searchsorted(cdf, u[:, None].to(cdf.dtype)).squeeze(-1)
    return tok.clamp_(max=probs.shape[-1] - 1)

In [ ]:
# Demo (synthetic): directions of three slots, and inverse-CDF sampling with one shared uniform stream.
slots = [("T1", "W1", 0), ("T1", "W1", 1), ("T2", "W3", 2)]
U_dirs = np.stack([pn_dir(*s) for s in slots])
print("unit norms:", np.linalg.norm(U_dirs, axis=1).round(4))
print("pairwise cosines (independent random directions in 4,096-d are almost orthogonal):\n", (U_dirs @ U_dirs.T).round(3))
us = u_stream("T1", "W1", 0)
print("first five token uniforms of slot T1|W1|0:", us[:5].round(4))

g = torch.Generator().manual_seed(0)
logits = torch.randn(1, 50, generator=g, dtype=torch.float64) * 3                      # a toy next-token distribution
p_twin = torch.softmax(logits, -1).expand(256, -1)
uu = torch.tensor(u_stream("T1", "W1", 0, n=256), dtype=torch.float64)                 # 256 shared uniforms
for eps in (0.0, 0.1, 0.5, 2.0):                                                       # growing logit perturbation
    p_edit = torch.softmax(logits + eps * torch.randn(1, 50, generator=g, dtype=torch.float64), -1).expand(256, -1)
    same = (_inverse_cdf(p_twin, uu) == _inverse_cdf(p_edit, uu)).double().mean().item()
    print(f"logit noise {eps:>4}: same token as the twin on {same:.0%} of 256 shared uniforms")

**2b. The displacement-targeted edit (`src/hooks.py`).** At every hooked block `l` and position, the code computes

$$h \leftarrow h + \big(\text{target} - (h - h_{\text{twin}})\cdot u\big)\,u$$

in fp32 and casts the result back to bf16.

- **What it sets.** The displacement *from the clean twin* along `u` equals `target` exactly. It does not compound
  across blocks, and the components orthogonal to `u` are left untouched.
- **The target.** The per-position target is `alpha × unit`. The unit is the norm of the displacement that a real
  *ban* prompt causes at that block and position (`||h_ban - h_twin||`), so every site gets the same
  "one ban's worth" of noise per position.
- **Where it goes.** S0, S4 and S5 edit the task-spec positions S, S1 edits R, and S2 edits COND. S3 is applied
  during decoding, against teacher-forced shadow rows.

In [ ]:
def make_prefill_edits(blocks: list[int], idx: torch.Tensor, u: dict[int, torch.Tensor],
                       target: dict[int, torch.Tensor], h_ref: dict[int, torch.Tensor]
                       ) -> dict[int, Callable[[torch.Tensor], torch.Tensor]]:
    """idx: LongTensor of positions; u: {l: fp32 [D] unit}; target: {l: fp32 [len(idx)]};
    h_ref: {l: fp32 [len(idx), D]} (clean twin block outputs at idx)."""
    edits = {}
    for l in blocks:
        def f(h: torch.Tensor, l: int = l) -> torch.Tensor:
            hs = h[:, idx].float()                       # [1, n, D]
            ul = u[l]
            comp = (hs - h_ref[l]) @ ul                  # [1, n]
            coef = target[l] - comp
            new = hs + coef[..., None] * ul
            h = h.clone()
            h[:, idx] = new.to(h.dtype)
            return h
        edits[l] = f
    return edits


def apply_edit_tensor(h: torch.Tensor, idx: torch.Tensor, ul: torch.Tensor, target: torch.Tensor,
                      h_ref: torch.Tensor) -> torch.Tensor:
    """Pure-tensor version of one block's edit (used by the CPU unit tests)."""
    hs = h[:, idx].float()
    coef = target - (hs - h_ref) @ ul
    out = h.clone()
    out[:, idx] = (hs + coef[..., None] * ul).to(h.dtype)
    return out

## 3. Pricing diversity: rarefied Hill numbers and the randomisation hull (`src/hull.py`)

This code was committed in #1, before any decisive sample was drawn.

- **Rarefied Hill-1.** Take the mean of `exp(Shannon entropy)` of the algorithm-label composition over subsamples of
  25 *correct* programs. A Hill of 1.0 means the cell always writes one algorithm, and 2.0 means two equally common
  ones.
- **The hull of one (task, site).** The settings are {twin (alpha 0), alpha 1, alpha 2, alpha 4}. Any mixture `w` of
  them is a valid sampling policy. Its cost is `pass_twin - Σ w_j pass_j`, and its Hill is estimated by a two-stage
  CRN draw.
- **Prices.** **H5** is the best Hill at a cost of at most 0.05. **P15** and **P20** are the cheapest costs that
  reach Hill 1.5 and 2.0 (`inf` = never reached).
- **The C1-site rule.** CONFIRMED if S0 has the best H5 (rounded to 0.01; ties go to the lower P15) on at least 4
  of 6 tasks. DISCONFIRMED if some other site has H5 ≥ H5(S0) on at least 4 tasks. Otherwise PARTIAL.

In [ ]:
DEPTH = 25
GRID_STEP = 0.05
COST_BOUND = 0.05
HILL_TARGETS = (1.5, 2.0)


def hill1(counts: np.ndarray) -> np.ndarray:
    """exp(Shannon entropy) along the last axis of a count array."""
    c = np.asarray(counts, dtype=np.float64)
    n = c.sum(-1, keepdims=True)
    p = np.where(n > 0, c / np.maximum(n, 1e-12), 0.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        h = -np.where(p > 0, p * np.log(p), 0.0).sum(-1)
    return np.exp(h)


def rarefied_hill(labels: list[str], depth: int = DEPTH, n_sub: int = 1000, seed: int = 0) -> dict[str, Any]:
    """Rarefied Hill-1; cells with fewer than `depth` correct programs report Hill at their own depth (flagged)."""
    n = len(labels)
    if n == 0:
        return {"hill": None, "depth": 0, "flag_own_depth": True, "n": 0}
    cats, inv = np.unique(np.array(labels), return_inverse=True)
    counts = np.bincount(inv, minlength=len(cats))
    m = min(depth, n)
    rng = np.random.default_rng(seed)
    sub = rng.multivariate_hypergeometric(counts, m, size=n_sub)
    return {"hill": float(hill1(sub).mean()), "depth": int(m), "flag_own_depth": bool(m < depth), "n": int(n)}


def simplex_grid(k: int, step: float = GRID_STEP, max_nonzero: int = 3) -> np.ndarray:
    m = int(round(1 / step))
    pts = []
    for comb in itertools.product(range(m + 1), repeat=k - 1):
        s = sum(comb)
        if s <= m:
            w = list(comb) + [m - s]
            if sum(1 for x in w if x > 0) <= max_nonzero:
                pts.append(w)
    return np.array(pts, dtype=np.float64) / m


class HullSampler:
    """Pre-draws the CRN randomness for one (task, site): per draw d, 25 uniforms (multinomial allocation by inverse
    CDF on pi(w)) and, per setting j, an ordered sequence of 25 correct programs (a random permutation, then draws with
    replacement beyond N_correct_j). C[j][d, n, :] = label counts among the first n programs of setting j in draw d."""

    def __init__(self, settings: list[dict[str, Any]], n_draws: int = 1000, depth: int = DEPTH, seed: int = 0):
        self.settings = settings
        self.depth = depth
        rng = np.random.default_rng(seed)
        cats = sorted({l for s in settings for l in s["correct_labels"]})
        self.cats = cats
        ci = {c: i for i, c in enumerate(cats)}
        K = max(len(cats), 1)
        self.U = rng.random((n_draws, depth))
        self.C = []
        self.with_replacement = []
        for s in settings:
            labs = np.array([ci[l] for l in s["correct_labels"]], dtype=np.int64)
            C = np.zeros((n_draws, depth + 1, K), dtype=np.int32)
            if len(labs):
                seqs = np.empty((n_draws, depth), dtype=np.int64)
                for d in range(n_draws):
                    if len(labs) >= depth:
                        seqs[d] = rng.permutation(labs)[:depth]
                    else:
                        extra = rng.choice(labs, size=depth - len(labs), replace=True)
                        seqs[d] = np.concatenate([rng.permutation(labs), extra])
                oh = np.zeros((n_draws, depth, K), dtype=np.int32)
                np.put_along_axis(oh, seqs[..., None], 1, axis=2)
                C[:, 1:] = np.cumsum(oh, axis=1)
            self.C.append(C)
            self.with_replacement.append(bool(0 < len(labs) < depth))
        self.pass_ = np.array([s["pass"] for s in settings], dtype=np.float64)
        self.n_correct = np.array([len(s["correct_labels"]) for s in settings])

    def hill_of(self, w: np.ndarray) -> float:
        a = w * self.pass_
        if a.sum() <= 0:
            return float("nan")
        pi = a / a.sum()
        cdf = np.cumsum(pi)
        cdf[-1] = 1.0
        j_of = np.searchsorted(cdf, self.U, side="right").clip(max=len(pi) - 1)  # [D, depth]
        comp = 0
        for j in range(len(pi)):
            nj = (j_of == j).sum(1)
            if pi[j] > 0:
                comp = comp + np.take_along_axis(self.C[j], nj[:, None, None], axis=1)[:, 0]
        return float(hill1(comp).mean())

    def cost_of(self, w: np.ndarray) -> float:
        return float(self.pass_[0] - (w * self.pass_).sum())


def hull_prices(settings: list[dict[str, Any]], n_draws: int = 1000, step: float = GRID_STEP, seed: int = 0,
                keep_frontier: bool = False) -> dict[str, Any]:
    """settings[0] must be the twin (alpha 0). Returns H5, P15, P20, the arg-max mixture and (optionally) the
    frontier (cost, Hill) points."""
    hs = HullSampler(settings, n_draws=n_draws, seed=seed)
    grid = simplex_grid(len(settings), step)
    costs = np.array([hs.cost_of(w) for w in grid])
    hills = np.array([hs.hill_of(w) for w in grid])
    ok = np.isfinite(hills)
    feas = ok & (costs <= COST_BOUND + 1e-12)
    h5 = float(hills[feas].max()) if feas.any() else float("nan")
    arg = grid[feas][int(np.argmax(hills[feas]))].tolist() if feas.any() else None
    out: dict[str, Any] = {"H5": h5, "H5_mixture": arg}
    for tgt in HILL_TARGETS:
        m = ok & (hills >= tgt)
        out[f"P{int(tgt * 10)}"] = float(costs[m].min()) if m.any() else math.inf
    out["with_replacement_flags"] = hs.with_replacement
    if keep_frontier:
        order = np.argsort(costs)
        fr, best = [], -1.0
        for i in order:
            if ok[i] and hills[i] > best:
                best = hills[i]
                fr.append([float(costs[i]), float(hills[i])])
        out["frontier"] = fr
    return out


# ------------------------------------------------------------------------------------------------ C1-site rule
def binom_tail(count: int, n: int = 6, p: float = 1 / 6) -> float:
    return float(sum(math.comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(count, n + 1)))


def best_site(h5: dict[str, float], p15: dict[str, float]) -> tuple[str | None, bool]:
    """argmax H5 rounded to 0.01; ties broken by lower P15 ('not reached' = inf); a remaining tie -> (None, True)."""
    vals = {s: round(v, 2) for s, v in h5.items() if v is not None and np.isfinite(v)}
    if not vals:
        return None, False
    top = max(vals.values())
    cand = [s for s, v in vals.items() if v == top]
    if len(cand) == 1:
        return cand[0], False
    pmin = min(p15.get(s, math.inf) for s in cand)
    cand2 = [s for s in cand if p15.get(s, math.inf) == pmin]
    if len(cand2) == 1:
        return cand2[0], False
    return None, True


def c1_decision(h5_by_task: dict[str, dict[str, float]], p15_by_task: dict[str, dict[str, float]]) -> dict[str, Any]:
    """Frozen rule: CONFIRMED if S0 is best on >= 4 of 6 tasks (raw p = P(X >= count | Bin(6, 1/6)));
    DISCONFIRMED (generic perturbation) if one other site has H5 >= H5(S0) on >= 4 of 6 tasks; otherwise PARTIAL."""
    best = {t: best_site(h5_by_task[t], p15_by_task[t]) for t in h5_by_task}
    n_t = len(h5_by_task)
    count = sum(1 for t, (b, _) in best.items() if b == "S0")
    others = sorted({s for t in h5_by_task for s in h5_by_task[t] if s != "S0"})
    ge = {s: sum(1 for t in h5_by_task if h5_by_task[t].get(s) is not None and h5_by_task[t].get("S0") is not None
                 and np.isfinite(h5_by_task[t][s]) and round(h5_by_task[t][s], 2) >= round(h5_by_task[t]["S0"], 2))
          for s in others}
    thr = 4 if n_t == 6 else math.ceil(2 * n_t / 3)
    if count >= thr:
        verdict = "CONFIRMED"
    elif any(v >= thr for v in ge.values()):
        verdict = "DISCONFIRMED (generic perturbation)"
    else:
        verdict = "PARTIAL"
    return {"best_site": {t: b for t, (b, _) in best.items()}, "unresolved_tie": {t: tie for t, (_, tie) in
                                                                                  best.items()},
            "count_S0_best": count, "n_tasks": n_t, "threshold": thr, "raw_p_binom": binom_tail(count, n_t, 1 / 6),
            "other_site_ge_S0_counts": ge, "verdict": verdict}

## 4. CPU unit tests (`tests/test_cpu.py`, stage `method.py tests-cpu`)

The original stage runs four tests:

- **T1, hook maths.** On synthetic tensors: the along-`u` error, the orthogonal components, identity at alpha 0, the
  positions outside the edit, and batch vs loop.
- **T2, hull.** An equal mix of two pure settings must give H5 ≈ 2.0, H5 must be monotone in the cost bound, and a
  pure setting must equal the single-arm rarefied Hill.
- **T3.** The two-survivor permutation null.
- **T5.** The binomial p for 4 of 6.

The notebook runs T1 in its *synthetic-scale* variant (`realistic=False`), because the realistic variant reads
`units/units_T03.npz`. It skips T4, the screen-file byte-identity test, because that test needs the earlier
artifacts' raw files. In `t2_hull`, the original changed `hull.COST_BOUND` through `import hull as H`. Here
`hull.py` lives in the notebook namespace, so the same change goes through `globals()`.

In [ ]:
def t1_hook(realistic: bool = True) -> dict:
    """realistic: residual norms and per-position targets taken from units/units_T03.npz at block 18 (the scales the
    GPU edits actually see); stress: targets 1-11 on |h| ~ 190 (reported, not gated)."""
    g = torch.Generator().manual_seed(0)
    D, P = 4096, 40
    idx = torch.arange(5, 30)
    if realistic:
        from cfg import UNITS_DIR  # notebook: unavailable here (needs the artifact's units/ folder); call with False

        z = np.load(UNITS_DIR / "units_T03.npz")
        scale = float(np.median(z["twin_norm_S"][18])) / np.sqrt(D)
        tgt = torch.tensor(z["nS"][18][:len(idx)], dtype=torch.float32)
    else:
        scale = 3.0
        tgt = torch.rand(len(idx), generator=g) * 10 + 1
    h_ref = torch.randn(P, D, generator=g) * scale
    h = (h_ref + torch.randn(P, D, generator=g) * scale / 6).to(torch.bfloat16)[None]
    u = torch.randn(D, generator=g)
    u = u / u.norm()
    out = apply_edit_tensor(h, idx, u, tgt, h_ref[idx])
    along = (out[0, idx].float() - h_ref[idx]) @ u
    rel = float(((along - tgt).abs() / tgt).max())
    # orthogonal components unchanged (up to the bf16 cast)
    d_before = h[0, idx].float() - h_ref[idx]
    d_after = out[0, idx].float() - h_ref[idx]
    orth_b = d_before - (d_before @ u)[:, None] * u
    orth_a = d_after - (d_after @ u)[:, None] * u
    orth_err = float(((orth_a - orth_b).norm(dim=-1) / orth_b.norm(dim=-1)).max())
    # alpha = 0 on the twin itself is the identity
    tw = h_ref.to(torch.bfloat16)[None]
    ident = torch.equal(apply_edit_tensor(tw, idx, u, torch.zeros(len(idx)), tw[0, idx].float()), tw)
    # positions outside idx untouched
    outside = torch.equal(out[0, :5], h[0, :5]) and torch.equal(out[0, 30:], h[0, 30:])
    # per-row batching equals the batch-1 loop
    hb = torch.cat([h, h], 0)
    ob = apply_edit_tensor(hb, idx, u, tgt, h_ref[idx])
    batch_eq = torch.equal(ob[0], out[0]) and torch.equal(ob[1], out[0])
    return {"max_rel_err_along_u_bf16": rel, "orthogonal_rel_change_max": orth_err, "identity_alpha0": ident,
            "outside_untouched": outside, "batch_equals_loop": batch_eq,
            "pass": rel < 1e-2 and ident and outside and batch_eq and orth_err < 2e-2}


def t2_hull() -> dict:
    s = [{"pass": 1.0, "correct_labels": ["A"] * 30},
         {"pass": 1.0, "correct_labels": ["B"] * 30}]
    r = hull_prices(s, n_draws=400, step=0.05)
    mono = []
    _COST_BOUND = COST_BOUND  # notebook (original: `from hull import COST_BOUND; import hull as H`)

    for b in (0.0, 0.05, 0.2):
        globals()["COST_BOUND"] = b  # original: H.COST_BOUND = b
        s2 = [{"pass": 1.0, "correct_labels": ["A"] * 30}, {"pass": 0.8, "correct_labels": ["B"] * 20 + ["C"] * 4},
              {"pass": 0.5, "correct_labels": ["D"] * 15}]
        mono.append(hull_prices(s2, n_draws=300, step=0.1)["H5"])
    globals()["COST_BOUND"] = _COST_BOUND  # original: H.COST_BOUND = COST_BOUND
    # lambda = 1 (pure single setting) equals the single-arm rarefied Hill
    labs = ["A"] * 12 + ["B"] * 8 + ["C"] * 10
    single = rarefied_hill(labs, depth=25, n_sub=4000)["hill"]
    pure = hull_prices([{"pass": 1.0, "correct_labels": labs}, {"pass": 1.0, "correct_labels": ["Z"] * 30}],
                       n_draws=4000, step=0.5)

    hs = HullSampler([{"pass": 1.0, "correct_labels": labs}, {"pass": 1.0, "correct_labels": ["Z"] * 30}],
                     n_draws=4000)
    h_pure = hs.hill_of(np.array([1.0, 0.0]))
    return {"equal_mix_H5": r["H5"], "equal_mix_mixture": r["H5_mixture"], "H5_monotone_in_bound": mono,
            "single_rarefied": single, "hull_pure_setting": h_pure,
            "pass": abs(r["H5"] - 2.0) < 0.1 and mono[0] <= mono[1] <= mono[2] and abs(single - h_pure) < 0.03}


def t3_perm() -> dict:
    d = {"bibfs": 1.0, "idastar": 3.0}
    q = {"bibfs": 0.7, "idastar": 0.3}
    obs = sum(q[y] * d[y] for y in q)
    vals = []
    for perm in itertools.permutations(list(d.values())):
        dd = dict(zip(d.keys(), perm))
        vals.append(sum(q[y] * dd[y] for y in q))
    p_edge = float(np.mean([v <= obs + 1e-12 for v in vals]))
    return {"n_perm": len(vals), "p_edge": p_edge, "pass": len(vals) == 2 and p_edge in (0.5, 1.0)}


def t5_binom() -> dict:
    p = binom_tail(4, 6, 1 / 6)
    return {"p_4_of_6": p, "pass": abs(p - 0.0087) < 5e-4}

In [ ]:
res = {"utc": utc_now(), "T1_hook_synthetic_scale": t1_hook(False), "T2_hull": t2_hull(), "T3_perm": t3_perm(),
       "T5_binom": t5_binom()}
ref_t = REF["unit_tests_cpu"]
published = {"T1_hook_synthetic_scale": ref_t["T1_hook_stress_small_targets"], "T2_hull": ref_t["T2_hull"],
             "T3_perm": ref_t["T3_perm"], "T5_binom": ref_t["T5_binom"]}
for name in ("T1_hook_synthetic_scale", "T2_hull", "T3_perm", "T5_binom"):
    print(f"{name}: pass = {res[name]['pass']}")
    for k, v in res[name].items():
        if k != "pass":
            print(f"    {k:28s} demo {v!s:42.42s} | published {published[name].get(k)!s:.42s}")

## 5. Load the labelled programs (`analysis.load_rows`)

The original joins three files by row key: `data/samples_raw.jsonl` (generation), `data/labels.jsonl` (labels and
code) and `data/row_logs.jsonl` (displacement logs). In the demo data, every slot carries the same fields under
`metadata_rows`, one entry per arm. The loop body that turns them into analysis records is the original, including
the D13 rule: a program with no `def solve` or with a syntax error counts as *invalid* for every task.

Each record holds the following fields:

| Field | Meaning |
|---|---|
| `attempt` | the program is a real attempt: not invalid, unresolved or disputed |
| `correct` | it is an attempt and it passed the tests |
| `off` | it is an attempt and its label differs from the task's default |
| `disp35` | the realised displacement at R in block 35, relative to the clean twin |

In [ ]:
INVALID = ("invalid",)
UNRES = ("unresolved", "disputed")
GRID_ALPHAS = (1.0, 2.0, 4.0)
TOP5 = ("bfs", "dfs", "gbfs", "iddfs", "astar")
SURV7 = ("bibfs", "idastar")
SURV_EXT = ("bibfs", "dijkstra_unit", "idastar", "jps", "biastar")


# ------------------------------------------------------------------------------------------------ data
def load_rows() -> list[dict[str, Any]]:
    # notebook: rebuild the three joined sources from the loaded demo data (one `metadata_rows` entry per arm)
    raw, labs, logs = [], {}, {}
    for ex in data["datasets"][0]["examples"]:
        if ex["metadata_k"] not in DEMO_K:
            continue
        for arm, x in ex["metadata_rows"].items():
            r = {"task": ex["metadata_task"], "template": ex["metadata_template"], "k": ex["metadata_k"], "arm": arm,
                 "site": x["site"], "alpha": x["alpha"], "finish_reason": x["finish_reason"],
                 "n_new_tokens": x["n_new_tokens"]}
            raw.append(r)
            labs[f"{r['task']}|{r['template']}|{r['arm']}|{r['k']}"] = x
            logs[f"{r['task']}|{r['template']}|{r['arm']}|{r['k']}"] = x
    k7 = {}  # data/labels_k7.jsonl (exp9 K=7 labels) only covers the C3d T1 rows, which are not in this subset
    out = []
    for r in raw:
        if r["arm"].startswith("SMOKE_"):
            continue
        key = f"{r['task']}|{r['template']}|{r['arm']}|{r['k']}"
        lb = labs.get(key)
        if lb is None:
            continue
        lg = logs.get(key, {})
        lab = lb["label"]
        if lab not in INVALID + UNRES and not _defines_solve(lb.get("code")):
            lab = "invalid"  # D13: the bench rule (no def solve / syntax error -> invalid) applied to every task
        d = DEFAULT[r["task"]]
        att = lab not in INVALID + UNRES
        rec = {"key": key, "task": r["task"], "w": r["template"], "k": int(r["k"]), "arm": r["arm"],
               "site": r["site"], "alpha": float(r["alpha"]), "label": lab, "passed": bool(lb.get("passed")),
               "attempt": att, "invalid": lab in INVALID, "unresolved": lab in UNRES,
               "correct": bool(lb.get("passed")) and att, "off": att and lab != d,
               "code_sha256": lb.get("code_sha256"), "code": lb.get("code"), "fallback_label": lb.get("fallback_label"),
               "disp35": (lg.get("realised_disp_R") or {}).get("35"), "ftkl": lg.get("first_token_kl"),
               "s3kl": ((lg.get("s3") or {}).get("kl_mean_steps1_32")), "finish": r["finish_reason"],
               "n_tokens": r["n_new_tokens"]}
        if key in k7:
            rec["label_k7"] = k7[key]["label_k7"]
            rec["label_ext"] = k7[key]["label_ext"]
            rec["passed_k7"] = k7[key]["passed"]
        out.append(rec)
    return out


def _defines_solve(code: str | None) -> bool:
    if not code or "def solve" not in code:
        return False
    try:
        tree = ast.parse(code)
    except (SyntaxError, ValueError):
        return False
    return any(isinstance(n, ast.FunctionDef) and n.name == "solve" for n in ast.walk(tree))


def cell(rows: list[dict[str, Any]], task: str, site: str, alpha: float, ks: list[int] | None) -> list[dict]:
    if site == "TWIN":
        return [r for r in rows if r["task"] == task and r["arm"] == "TWIN" and (ks is None or r["k"] in ks)]
    return [r for r in rows if r["task"] == task and r["site"] == site and r["alpha"] == alpha and
            r["arm"] != "TWIN_X" and (ks is None or r["k"] in ks)]


def grid_ks(site: str, alpha: float) -> list[int]:
    return N50_K if (site == "TWIN" or alpha == 8.0 or site == "TOP5") else N_GRID_K

In [ ]:
setup_logging("analysis")
t_start = time.time()
rows = load_rows()
logger.info(f"loaded {len(rows)} labelled program rows")
df_rows = pd.DataFrame(rows)
for t in TASKS:
    print(f"{t} (default: {DEFAULT[t]}) -- algorithm labels of all programs, alpha 1-4 pooled per site")
    for s in ("TWIN",) + SITES + ("S1E", "S2E"):
        cnt = Counter(df_rows[(df_rows.task == t) & (df_rows.site == s)]["label"])
        print(f"   {s:4s} " + ", ".join(f"{k} {v}" for k, v in cnt.most_common()))

## 6. Cells (STEP 11.1)

A *cell* is a (task, site, alpha) group of programs. `cell_stats` computes, for each cell:

- the pass, invalid and unresolved rates;
- the attempted and correct label compositions;
- the off-default shares, among attempts and among correct programs;
- the rarefied correct-only Hill;
- the excess over the twin and the pass ratio;
- the phrasing clusters per idea: correct programs with the same label, grouped by Jaccard similarity of AST
  4-shingles at θ = 0.7, 0.8 and 0.9;
- the median realised displacement.

The phrasing clusters separate *surface* variation (the same idea, written another way) from *idea* variation (a
different algorithm).

In [ ]:
def _ast_tokens(code: str) -> list[str]:
    try:
        tree = ast.parse(code)
    except (SyntaxError, ValueError):
        return []
    toks: list[str] = []

    def visit(n: ast.AST) -> None:
        name = type(n).__name__
        if isinstance(n, (ast.Name, ast.arg)):
            toks.append("ID")
        elif isinstance(n, ast.Constant):
            toks.append("C:" + type(n.value).__name__)
        elif isinstance(n, ast.Attribute):
            toks.append("ATTR:" + n.attr)
        elif isinstance(n, ast.FunctionDef):
            toks.append("FUNC")
        else:
            toks.append(name)
        for c in ast.iter_child_nodes(n):
            visit(c)
    visit(tree)
    return toks


def shingles(code: str, n: int = 4) -> frozenset:
    t = _ast_tokens(code or "")
    return frozenset(tuple(t[i:i + n]) for i in range(max(0, len(t) - n + 1)))


def clusters_per_idea(rows: list[dict[str, Any]], theta: float) -> dict[str, Any]:
    by: dict[str, list[frozenset]] = defaultdict(list)
    for r in rows:
        if r["correct"] and r.get("code"):
            by[r["label"]].append(shingles(r["code"]))
    per = {}
    for lab, sh in by.items():
        n = len(sh)
        parent = list(range(n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        for i in range(n):
            for j in range(i + 1, n):
                a, b = sh[i], sh[j]
                u = len(a | b)
                if u and len(a & b) / u >= theta:
                    parent[find(i)] = find(j)
        per[lab] = {"n": n, "clusters": len({find(i) for i in range(n)})}
    tot_c = sum(v["clusters"] for v in per.values())
    tot_n = sum(v["n"] for v in per.values())
    return {"per_idea": per, "clusters_per_idea": tot_c / len(per) if per else None,
            "clusters_per_10_programs": 10 * tot_c / tot_n if tot_n else None}


def cell_stats(rs: list[dict[str, Any]], default: str, twin: dict[str, Any] | None = None,
               clusters: bool = True) -> dict[str, Any]:
    n = len(rs)
    att = [r for r in rs if r["attempt"]]
    cor = [r for r in rs if r["correct"]]
    out: dict[str, Any] = {"n": n, "n_attempted": len(att), "n_correct": len(cor)}
    out["pass_rate"] = sum(r["passed"] for r in rs) / n if n else None
    out["pass_ci"] = wilson_ci(sum(r["passed"] for r in rs), n) if n else None
    out["invalid_rate"] = sum(r["invalid"] for r in rs) / n if n else None
    out["unresolved_rate"] = sum(r["unresolved"] for r in rs) / n if n else None
    out["other_rate"] = sum(r["label"] == "other" for r in att) / len(att) if att else None
    out["attempted_composition"] = dict(Counter(r["label"] for r in att))
    out["correct_composition"] = dict(Counter(r["label"] for r in cor))
    out["off_share_attempts"] = sum(r["off"] for r in att) / len(att) if att else None
    out["off_share_correct"] = sum(r["label"] != default for r in cor) / len(cor) if cor else None
    h = rarefied_hill([r["label"] for r in cor], depth=DEPTH)
    out["hill_correct"] = h["hill"]
    out["hill_depth"] = h["depth"]
    out["hill_own_depth_flag"] = h["flag_own_depth"]
    h2 = rarefied_hill([r["label"] for r in cor if r["label"] != "other"], depth=DEPTH)
    out["hill_correct_excl_other"] = h2["hill"]
    out["finish_reasons"] = dict(Counter(r["finish"] for r in rs))
    if twin is not None:
        out["excess_correct_off"] = (out["off_share_correct"] - twin["off_share_correct"]) \
            if out["off_share_correct"] is not None and twin.get("off_share_correct") is not None else None
        out["excess_attempt_off"] = (out["off_share_attempts"] - twin["off_share_attempts"]) \
            if out["off_share_attempts"] is not None and twin.get("off_share_attempts") is not None else None
        out["pass_ratio"] = out["pass_rate"] / twin["pass_rate"] if twin.get("pass_rate") else None
    if clusters:
        out["phrasing"] = {str(th): clusters_per_idea(rs, th) for th in (0.7, 0.8, 0.9)}
    d35 = [r["disp35"] for r in rs if r.get("disp35") is not None]
    out["realised_disp_R35_median"] = float(np.median(d35)) if d35 else None
    s3 = [r["s3kl"] for r in rs if r.get("s3kl") is not None]
    out["s3_kl_steps1_32_median"] = float(np.median(s3)) if s3 else None
    return out


def all_cells(rows: list[dict[str, Any]]) -> dict[str, Any]:
    res: dict[str, Any] = {}
    for t in TASKS:
        d = DEFAULT[t]
        tw = cell_stats(cell(rows, t, "TWIN", 0.0, N50_K), d)
        res[f"{t}|TWIN|0"] = tw
        twx = [r for r in rows if r["task"] == t and r["arm"] == "TWIN_X"]
        if twx:
            res[f"{t}|TWIN_X|0"] = cell_stats(twx, d, tw, clusters=False)
        for s in SITES:
            for a in GRID_ALPHAS:
                rs = cell(rows, t, s, a, N_GRID_K)
                if rs:
                    res[f"{t}|{s}|{a:g}"] = cell_stats(rs, d, tw)
        rs = cell(rows, t, "S0", 8.0, N50_K)
        if rs:
            res[f"{t}|S0|8"] = cell_stats(rs, d, tw)
        for s in ("S1E", "S2E"):
            rs = cell(rows, t, s, 1.0, N_GRID_K)
            if rs:
                res[f"{t}|{s}|1"] = cell_stats(rs, d, tw)
        if t == "T1":
            for a in GRID_ALPHAS:
                rs = cell(rows, t, "TOP5", a, N50_K)
                if rs:
                    res[f"{t}|TOP5|{a:g}"] = cell_stats(rs, d, tw)
                rs = cell(rows, t, "S0", a, N50_K)
                if rs:
                    res[f"{t}|S0_N50|{a:g}"] = cell_stats(rs, d, tw)
        for eng in ("ENG",):
            rs = [r for r in rows if r["task"] == t and r["site"] == eng]
            if rs:
                res[f"{t}|ENG|1"] = cell_stats(rs, d, tw)
    return res

In [ ]:
cells = all_cells(rows)
tab = pd.DataFrame({k: {"n": v["n"], "pass": v["pass_rate"], "invalid": v["invalid_rate"],
                        "off_share_correct": v["off_share_correct"], "hill_correct": v["hill_correct"],
                        "correct_composition": v["correct_composition"]} for k, v in cells.items()}).T
print(f"{len(cells)} cells; the twin and the alpha-2 cells:")
with pd.option_context("display.max_colwidth", 70, "display.width", 200):
    print(tab[[("|TWIN|" in k or k.endswith("|2")) for k in tab.index]].round(3).to_string())

## 7. Randomisation hull per (task, site) + template-stratified bootstrap (STEP 11.2)

For each task and site, the settings are {twin, alpha 1, 2, 4}. The code computes:

- **The point estimate.** H5, P15 and P20, plus the frontier for the figure.
- **Sensitivities.** Mixtures of *adjacent* doses only. (The S0 + alpha-8 sensitivity also exists, but the
  alpha-8 rows belong to another dataset and are not in this subset.)
- **The bootstrap.** Programs are resampled within template, the twin once per replicate and shared by every
  site, and the whole hull, including the max over mixtures, is recomputed inside each replicate. That is
  "selection inside the bootstrap".

**Notebook changes.** The original fans the bootstrap out with
`ProcessPoolExecutor(max_workers=workers, mp_context=spawn)`. Spawned workers cannot import functions defined in a
notebook, so the notebook runs the same jobs with the same seeds through a plain `map`, which gives identical
numbers. The hard-coded draw counts and grid steps are the config variables.

In [ ]:
def hull_settings(rows: list[dict[str, Any]], t: str, s: str, alphas=GRID_ALPHAS) -> list[dict[str, Any]]:
    out = []
    for a in (0.0,) + tuple(alphas):
        rs = cell(rows, t, "TWIN" if a == 0.0 else s, a, grid_ks("TWIN" if a == 0 else s, a))
        out.append({"alpha": a, "rows": rs})
    return out


def _settings_from(rows_by_setting: list[list[dict[str, Any]]]) -> list[dict[str, Any]]:
    return [{"pass": (sum(r["passed"] for r in rs) / len(rs)) if rs else 0.0,
             "correct_labels": [r["label"] for r in rs if r["correct"]]} for rs in rows_by_setting]


def _strat_resample(rs: list[dict[str, Any]], rng: np.random.Generator) -> list[dict[str, Any]]:
    by: dict[str, list] = defaultdict(list)
    for r in rs:
        by[r["w"]].append(r)
    out = []
    for w in sorted(by):
        g = by[w]
        idx = rng.integers(0, len(g), size=len(g))
        out.extend(g[i] for i in idx)
    return out


def _boot_task(args: tuple) -> dict[str, Any]:
    t, packs, B, seed = args
    rng = np.random.default_rng(seed)
    out = {s: {"H5": [], "P15": [], "P20": []} for s in packs}
    for b in range(B):
        # the twin is resampled once per replicate and shared by every site (same twin in every hull)
        twin_b = _strat_resample(packs[next(iter(packs))][0], rng)
        for s, sets in packs.items():
            rb = [twin_b] + [_strat_resample(x, rng) for x in sets[1:]]
            hp = hull_prices(_settings_from(rb), n_draws=BOOT_N_DRAWS, step=BOOT_STEP, seed=int(rng.integers(1 << 31)))
            for key in ("H5", "P15", "P20"):
                out[s][key].append(hp[key])
    return {"task": t, "boot": out}


def hull_all(rows: list[dict[str, Any]], B: int, workers: int) -> dict[str, Any]:
    t0 = time.time()
    point: dict[str, Any] = {}
    packs_by_task = {}
    for t in TASKS:
        packs = {}
        for s in SITES:
            st = hull_settings(rows, t, s)
            if any(not x["rows"] for x in st):
                continue
            packs[s] = [x["rows"] for x in st]
            hp = hull_prices(_settings_from(packs[s]), n_draws=POINT_N_DRAWS, step=POINT_STEP, seed=0,
                             keep_frontier=True)
            point[f"{t}|{s}"] = hp
            # sensitivity 1: linear interpolation along alpha (mixtures of adjacent settings only)
            # sensitivity 2: S0 with alpha 8 appended
        if "S0" in packs:
            r8 = cell(rows, t, "S0", 8.0, N50_K)
            if r8:
                hp8 = hull_prices(_settings_from(packs["S0"] + [r8]), n_draws=POINT_N_DRAWS, step=POINT_STEP, seed=0,
                                  keep_frontier=True)
                point[f"{t}|S0+a8"] = hp8
        packs_by_task[t] = packs
    for t, packs in packs_by_task.items():
        for s, sets in packs.items():
            point[f"{t}|{s}"]["adjacent_only"] = _adjacent_interp(_settings_from(sets))
    logger.info(f"hull point estimates in {time.time() - t0:.0f}s")
    boot: dict[str, Any] = {}
    if B > 0:
        jobs = [(t, packs_by_task[t], B, 1000 + i) for i, t in enumerate(TASKS) if packs_by_task.get(t)]
        # notebook: in-process map instead of ProcessPoolExecutor(max_workers=workers, mp_context=spawn)
        for res in map(_boot_task, jobs):
            boot[res["task"]] = res["boot"]
        logger.info(f"hull bootstrap B={B} in {time.time() - t0:.0f}s")
    return {"point": point, "boot": boot, "B": B}


def _adjacent_interp(settings: list[dict[str, Any]]) -> dict[str, Any]:
    """Sensitivity: only mixtures of two adjacent settings along alpha (linear interpolation)."""
    hs = HullSampler(settings, n_draws=1000, seed=0)
    best = -1.0
    p15 = p20 = math.inf
    for j in range(len(settings) - 1):
        for lam in np.linspace(0, 1, 21):
            w = np.zeros(len(settings))
            w[j], w[j + 1] = 1 - lam, lam
            c, h = hs.cost_of(w), hs.hill_of(w)
            if not np.isfinite(h):
                continue
            if c <= COST_BOUND + 1e-12:
                best = max(best, h)
            if h >= 1.5:
                p15 = min(p15, c)
            if h >= 2.0:
                p20 = min(p20, c)
    return {"H5": best if best >= 0 else None, "P15": p15, "P20": p20}

In [ ]:
hull = hull_all(rows, B=BOOT_B, workers=WORKERS)

## 8. The C1-site decision (STEP 11.3)

The frozen rule applied to the point estimates gives, per task:

- the best site;
- **D_t** = H5(S0) − max over the other sites of H5, with its bootstrap 95% CI;
- the bootstrap probability that S0 is the *unique* arg-max.

The cell after the code puts these next to the published full-run values (N = 30 per cell, 2,000 replicates).

In [ ]:
def c1_site(hull: dict[str, Any]) -> dict[str, Any]:
    pt = hull["point"]
    h5 = {t: {s: pt[f"{t}|{s}"]["H5"] for s in SITES if f"{t}|{s}" in pt} for t in TASKS}
    p15 = {t: {s: pt[f"{t}|{s}"]["P15"] for s in SITES if f"{t}|{s}" in pt} for t in TASKS}
    h5 = {t: v for t, v in h5.items() if len(v) == len(SITES)}
    p15 = {t: p15[t] for t in h5}
    dec = c1_decision(h5, p15)
    if not h5:
        dec["verdict"] = "NOT EVALUABLE (no task has all six sites yet)"
    per: dict[str, Any] = {}
    for t in h5:
        others = {s: v for s, v in h5[t].items() if s != "S0"}
        D = h5[t]["S0"] - max(others.values())
        rec: dict[str, Any] = {"H5": h5[t], "P15": p15[t], "D_t": D,
                               "P20": {s: pt[f"{t}|{s}"]["P20"] for s in SITES}}
        bt = hull["boot"].get(t)
        if bt:
            Bn = len(bt["S0"]["H5"])
            arr = {s: np.array(bt[s]["H5"], dtype=float) for s in SITES if s in bt}
            Db = arr["S0"] - np.nanmax(np.stack([arr[s] for s in arr if s != "S0"]), axis=0)
            Db = Db[np.isfinite(Db)]
            rec["D_t_ci95"] = [float(np.quantile(Db, 0.025)), float(np.quantile(Db, 0.975))] if len(Db) else None
            present = [s for s in SITES if s in arr]
            M = np.stack([np.round(arr[s], 2) for s in present])
            win = np.zeros(Bn)
            for b in range(Bn):
                col = M[:, b]
                if np.all(~np.isfinite(col)):
                    continue
                mx = np.nanmax(col)
                cands = [present[i] for i in range(len(present)) if col[i] == mx]
                if cands == ["S0"]:
                    win[b] = 1
            rec["P_S0_argmax_boot"] = float(win.mean())
            rec["H5_ci95"] = {s: [float(np.nanquantile(arr[s], 0.025)), float(np.nanquantile(arr[s], 0.975))]
                              for s in arr}
            rec["S0_within_0.10_of_best_share_boot"] = float(np.mean(Db >= -0.10)) if len(Db) else None
            rec["close_call"] = bool(rec["D_t_ci95"] and rec["D_t_ci95"][0] < 0 < rec["D_t_ci95"][1])
        per[t] = rec
    dec["per_task"] = per
    dec["equivalence_margin_0.10"] = {t: bool(abs(per[t]["D_t"]) <= 0.10) for t in per}
    return dec

In [ ]:
c1 = c1_site(hull)
ref_c1 = REF["c1_site"]
print(f"DEMO      : verdict {c1['verdict']}; S0 best on {c1['count_S0_best']}/{c1['n_tasks']} tasks "
      f"(raw p = {c1['raw_p_binom']:.4f}); other site >= S0: {c1['other_site_ge_S0_counts']}")
print(f"FULL RUN  : verdict {ref_c1['verdict']}; S0 best on {ref_c1['count_S0_best']}/{ref_c1['n_tasks']} tasks "
      f"(raw p = {ref_c1['raw_p_binom']:.4f}); other site >= S0: {ref_c1['other_site_ge_S0_counts']}\n")
recs = []
for t in TASKS:
    d, f = c1["per_task"][t], ref_c1["per_task"][t]
    rec = {"task": f"{t} ({DEFAULT[t]})"}
    for s in SITES:
        rec[f"H5 {s}"] = f"{d['H5'][s]:.2f} ({f['H5'][s]:.2f})"
    rec["best"] = f"{c1['best_site'][t]} ({ref_c1['best_site'][t]})"
    ci = d.get("D_t_ci95") or [float("nan")] * 2
    rec["D_t [95% CI]"] = f"{d['D_t']:+.2f} [{ci[0]:+.2f}, {ci[1]:+.2f}]"
    rec["P(S0 argmax)"] = f"{d.get('P_S0_argmax_boot', float('nan')):.2f} ({f['P_S0_argmax_boot']:.2f})"
    recs.append(rec)
print("demo value (published full-run value):")
print(pd.DataFrame(recs).set_index("task").to_string())

## 9. Failure modes and Hill gain (STEPS 11.4-11.5)

The frozen per-(task, site) classification uses three quantities at each dose: the correct-only off-default
*excess* over the twin, the pass ratio relative to the twin, and the invalid rate.

- **moves:** excess ≥ 0.20 with pass ratio ≥ 0.8 at some dose.
- **moves with cost:** excess ≥ 0.10, but only where the pass ratio is below 0.8.
- **breakage first:** passes are lost, or invalid programs reach 0.2, before the excess reaches 0.10.
- **no movement:** neither of the above.
- **unclassified (weak movement):** whatever the frozen rule leaves uncovered.

The Hill gain is ΔHill per point of pass rate lost.

In [ ]:
def failure_modes(cells: dict[str, Any]) -> dict[str, Any]:
    out: dict[str, Any] = {}
    for t in TASKS:
        tw = cells.get(f"{t}|TWIN|0")
        if not tw:
            continue
        for s in SITES:
            rec = {}
            for a in GRID_ALPHAS:
                c = cells.get(f"{t}|{s}|{a:g}")
                if c:
                    rec[a] = {"excess": c["excess_correct_off"], "pass_ratio": c["pass_ratio"],
                              "invalid": c["invalid_rate"]}
            if not rec:
                continue
            ex = {a: (v["excess"] if v["excess"] is not None else 0.0) for a, v in rec.items()}
            pr = {a: (v["pass_ratio"] if v["pass_ratio"] is not None else 0.0) for a, v in rec.items()}
            iv = {a: v["invalid"] or 0.0 for a, v in rec.items()}
            if any(ex[a] >= 0.20 and pr[a] >= 0.8 for a in rec):
                mode = "moves"
            elif any(ex[a] >= 0.10 for a in rec) and all(pr[a] < 0.8 for a in rec if ex[a] >= 0.10):
                mode = "moves with cost"
            elif any((pr[a] < 0.8 or iv[a] >= 0.2) and ex[a] < 0.10 for a in rec):
                mode = "breakage first"
            elif all(ex[a] < 0.10 and pr[a] >= 0.8 for a in rec):
                mode = "no movement"
            else:
                mode = "unclassified (weak movement: 0.10 <= excess < 0.20 with pass >= 0.8 x twin)"
            out[f"{t}|{s}"] = {"mode": mode, "by_alpha": rec}
    summ = {s: dict(Counter(v["mode"] for k, v in out.items() if k.endswith("|" + s))) for s in SITES}
    return {"per_task_site": out, "summary_by_site": summ}


def hill_gain(cells: dict[str, Any]) -> dict[str, Any]:
    out = {}
    for t in TASKS:
        tw = cells.get(f"{t}|TWIN|0")
        if not tw:
            continue
        for s in SITES:
            for a in GRID_ALPHAS:
                c = cells.get(f"{t}|{s}|{a:g}")
                if not c or c["hill_correct"] is None or tw["hill_correct"] is None:
                    continue
                dp = tw["pass_rate"] - c["pass_rate"]
                out[f"{t}|{s}|{a:g}"] = {"G": (c["hill_correct"] - tw["hill_correct"]) / max(dp, 0.01),
                                         "dHill": c["hill_correct"] - tw["hill_correct"], "dPass": dp}
    return out

In [ ]:
fm = failure_modes(cells)
hg = hill_gain(cells)
modes = pd.DataFrame({s: {t: fm["per_task_site"][f"{t}|{s}"]["mode"].split(" (")[0] for t in TASKS} for s in SITES}).T
print("Failure mode per site (demo subset):")
print(modes.to_string())
print("\nsummary by site -- demo vs published:")
for s in SITES:
    print(f"  {s}: demo {fm['summary_by_site'][s]}\n      full {REF['failure_modes_summary_by_site'][s]}")

## 10. Paired flips under CRN (STEP 11.6)

Every arm on a slot shares the twin's random direction and token uniforms, so S0 and any other site can be compared
**slot by slot**. The 2×2 table counts how often a slot goes off default under both, under S0 only, under the other
site only, or under neither, and the exact McNemar test compares the two discordant counts. The byte-identical rate
is the share of slots where the other site's program is the twin's program, character for character, which happens
when the noise did nothing to the output.

In [ ]:
def mcnemar_exact(b: int, c: int) -> float:
    from scipy.stats import binomtest

    n = b + c
    return 1.0 if n == 0 else float(binomtest(min(b, c), n, 0.5).pvalue)


def paired_flips(rows: list[dict[str, Any]]) -> dict[str, Any]:
    idx = {(r["task"], r["w"], r["k"], r["arm"]): r for r in rows}
    out = {}
    for t in TASKS:
        for a in GRID_ALPHAS:
            s0a = f"S0_a{a:g}"
            for s in SITES[1:]:
                arm = f"{s}_a{a:g}"
                tab = {"attempts": [0, 0, 0, 0], "correct": [0, 0, 0, 0]}  # [both, S0 only, site only, neither]
                ident_twin = ident_s0 = npair = 0
                for w in TEMPLATE_IDS:
                    for k in N_GRID_K:
                        r0, r1, tw = idx.get((t, w, k, s0a)), idx.get((t, w, k, arm)), idx.get((t, w, k, "TWIN"))
                        if not r0 or not r1:
                            continue
                        npair += 1
                        if tw is not None:
                            ident_twin += (r1["code_sha256"] is not None and r1["code_sha256"] == tw["code_sha256"])
                        ident_s0 += (r1["code_sha256"] is not None and r1["code_sha256"] == r0["code_sha256"])
                        for kind, ok in (("attempts", r0["attempt"] and r1["attempt"]),
                                         ("correct", r0["correct"] and r1["correct"])):
                            if not ok:
                                continue
                            o0 = r0["label"] != DEFAULT[t]
                            o1 = r1["label"] != DEFAULT[t]
                            tab[kind][0 if (o0 and o1) else 1 if o0 else 2 if o1 else 3] += 1
                if npair == 0:
                    continue
                out[f"{t}|{s}|{a:g}"] = {
                    "n_pairs": npair, "table_attempts[both,S0only,siteonly,neither]": tab["attempts"],
                    "mcnemar_p_attempts": mcnemar_exact(tab["attempts"][1], tab["attempts"][2]),
                    "table_correct": tab["correct"], "mcnemar_p_correct": mcnemar_exact(tab["correct"][1],
                                                                                        tab["correct"][2]),
                    "byte_identical_vs_twin": ident_twin / npair, "byte_identical_vs_S0": ident_s0 / npair}
    # twin vs S0 byte-identical rates (inertness of S0 itself)
    for t in TASKS:
        for a in GRID_ALPHAS:
            n = same = 0
            for w in TEMPLATE_IDS:
                for k in N_GRID_K:
                    r0, tw = idx.get((t, w, k, f"S0_a{a:g}")), idx.get((t, w, k, "TWIN"))
                    if r0 and tw:
                        n += 1
                        same += (r0["code_sha256"] is not None and r0["code_sha256"] == tw["code_sha256"])
            if n:
                out[f"{t}|S0|{a:g}|vs_twin"] = {"n_pairs": n, "byte_identical_vs_twin": same / n}
    return out

In [ ]:
flips = paired_flips(rows)
recs = {}
for s in SITES[1:]:
    recs[s] = {}
    for t in TASKS:
        r = flips[f"{t}|{s}|2"]
        tb = r["table_attempts[both,S0only,siteonly,neither]"]
        recs[s][t] = f"{tb[1]}:{tb[2]} (p={r['mcnemar_p_attempts']:.2g}; id={r['byte_identical_vs_twin']:.2f})"
print(f"alpha 2 -- slots off default under S0 only : under the site only (exact McNemar p; byte-identical to twin), "
      f"{flips['T03|S1|2']['n_pairs']} paired slots per task")
print(pd.DataFrame(recs).T.to_string())

## 11. Transmission and the pooled model (STEPS 11.7-11.8)

Could S0 win only because it pushes the downstream states harder? S1 moves the read position R at block 35 just as
far as S0 does. The **transmission GLM** tests this with a logistic regression of *correct-only off-default* on log
realised displacement at R (block 35), with an S0 indicator and template and task fixed effects, over the prefill
sites. If S0's coefficient stays above zero at matched downstream displacement, the effect is site-specific rather
than generic transmission. Two more pieces sit in the same function:

- **Matched-dose bins.** The off-default share for S0 vs the other sites within quintiles of the displacement.
- **S3.** It is modelled on its own step-wise KL.

The **pooled model** is a Bayesian mixed logistic regression (statsmodels `BinomialBayesMixedGLM`, variational
Bayes) of off-default attempts on site × log2(alpha), with random task and template effects.

**Expected warnings.** statsmodels raises a `ConvergenceWarning` for the `disp_R35_sites` variant, which has one
dummy per site; the published full run flags the same fit as not converged. The first-token-KL model is
uninformative here (deviation D5): the first generated token is always the code fence, so the KL is below 1e-6 for
about 90% of rows.

In [ ]:
def transmission(rows: list[dict[str, Any]]) -> dict[str, Any]:
    import pandas as pd
    import statsmodels.formula.api as smf

    pre = [r for r in rows if r["site"] in ("S0", "S1", "S2", "S4", "S5", "S1E", "S2E") and r["correct"] and
           r.get("disp35") and r["k"] in N_GRID_K and r["alpha"] in (1.0, 2.0, 4.0)]
    df = pd.DataFrame([{"y": int(r["label"] != DEFAULT[r["task"]]), "logd": math.log(max(r["disp35"], 1e-6)),
                        "logkl": math.log(max(r["ftkl"] or 0.0, 1e-12)), "is_S0": int(r["site"] == "S0"),
                        "site": r["site"], "task": r["task"], "w": r["w"], "alpha": r["alpha"]} for r in pre])
    out: dict[str, Any] = {"n_rows": len(df)}
    if df.empty:
        return out
    var_tasks = [t for t, g in df.groupby("task") if 0 < g["y"].mean() < 1]
    d2 = df[df["task"].isin(var_tasks)].copy()
    out["tasks_with_variation"] = var_tasks
    out["n_rows_used"] = len(d2)
    out["ftkl_degenerate"] = bool(df["logkl"].std() < 1e-6 or (np.exp(df["logkl"]) < 1e-6).mean() > 0.9)
    out["ftkl_share_below_1e-6"] = float((np.exp(df["logkl"]) < 1e-6).mean())
    for name, f in (("disp_R35", "y ~ logd + is_S0 + C(w) + C(task)"),
                    ("disp_R35_sites", "y ~ logd + C(site, Treatment('S0')) + C(w) + C(task)"),
                    ("first_token_kl", "y ~ logkl + is_S0 + C(w) + C(task)")):
        try:
            if len(var_tasks) < 1 or len(d2) < 20:
                raise ValueError("too few rows with variation")
            m = smf.logit(f.replace(" + C(task)", "") if len(var_tasks) == 1 else f, data=d2).fit(disp=0,
                                                                                                    maxiter=200)
            ci = m.conf_int()
            out[name] = {"params": {k: float(v) for k, v in m.params.items() if not k.startswith("C(w")},
                         "ci95": {k: [float(ci.loc[k, 0]), float(ci.loc[k, 1])] for k in m.params.index
                                  if not k.startswith("C(w") and not k.startswith("C(task")},
                         "converged": bool(m.mle_retvals.get("converged", True)), "n": int(m.nobs)}
            if "is_S0" in m.params:
                lo, hi = ci.loc["is_S0", 0], ci.loc["is_S0", 1]
                out[name]["S0_reading"] = ("site-specific beyond downstream dose" if lo > 0 else
                                           "S0 below others at matched transmission" if hi < 0 else
                                           "generic transmission (S0 indicator CI includes 0)")
        except Exception as e:  # noqa: BLE001 - separation / singular fits are reported, not hidden
            out[name] = {"error": str(e)[:200]}
    # matched-dose bins: off-default among correct programs, S0 vs other prefill sites within quintiles of log d
    if len(df) >= 10:
        q = np.quantile(df["logd"], [0, 0.2, 0.4, 0.6, 0.8, 1.0])
        bins = []
        for i in range(5):
            g = df[(df["logd"] >= q[i]) & (df["logd"] <= q[i + 1])]
            a, b = g[g["is_S0"] == 1], g[g["is_S0"] == 0]
            bins.append({"logd_range": [float(q[i]), float(q[i + 1])], "n_S0": len(a), "n_other": len(b),
                         "off_S0": float(a["y"].mean()) if len(a) else None,
                         "off_other": float(b["y"].mean()) if len(b) else None})
        out["matched_dose_bins"] = bins
    # S3 on its own step-wise KL
    s3 = [r for r in rows if r["site"] == "S3" and r["correct"] and r.get("s3kl") is not None and
          r["k"] in N_GRID_K]
    if s3:
        d3 = pd.DataFrame([{"y": int(r["label"] != DEFAULT[r["task"]]), "logkl": math.log(max(r["s3kl"], 1e-9)),
                            "task": r["task"], "alpha": r["alpha"]} for r in s3])
        out["S3_stepwise_kl"] = {"n": len(d3), "off_rate": float(d3["y"].mean()),
                                 "median_kl_by_alpha": {str(a): float(g["logkl"].apply(math.exp).median())
                                                        for a, g in d3.groupby("alpha")}}
        try:
            vt = [t for t, g in d3.groupby("task") if 0 < g["y"].mean() < 1]
            d4 = d3[d3["task"].isin(vt)]
            if len(vt) >= 1 and len(d4) >= 20:
                m = smf.logit("y ~ logkl" + (" + C(task)" if len(vt) > 1 else ""), data=d4).fit(disp=0)
                out["S3_stepwise_kl"]["logkl_coef"] = float(m.params["logkl"])
                out["S3_stepwise_kl"]["logkl_ci95"] = [float(x) for x in m.conf_int().loc["logkl"]]
        except Exception as e:  # noqa: BLE001
            out["S3_stepwise_kl"]["error"] = str(e)[:200]
    # descriptive: median realised displacement at R (block 35) per site and alpha
    md = defaultdict(list)
    for r in rows:
        if r.get("disp35") is not None and r["k"] in N_GRID_K:
            md[f"{r['site']}|{r['alpha']:g}"].append(r["disp35"])
    out["median_disp_R35_by_site_alpha"] = {k: float(np.median(v)) for k, v in sorted(md.items())}
    return out


def pooled_model(rows: list[dict[str, Any]]) -> dict[str, Any]:
    import pandas as pd

    rs = [r for r in rows if r["site"] in SITES and r["attempt"] and r["k"] in N_GRID_K and r["alpha"] in GRID_ALPHAS]
    df = pd.DataFrame([{"y": int(r["off"]), "site": r["site"], "la": math.log2(r["alpha"]), "task": r["task"],
                        "w": r["w"]} for r in rs])
    out: dict[str, Any] = {"n": len(df)}
    if df.empty:
        return out
    try:
        from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

        m = BinomialBayesMixedGLM.from_formula("y ~ C(site, Treatment('S0')) * la",
                                               {"task": "0 + C(task)", "template": "0 + C(w)"}, df)
        f = m.fit_vb()
        names = m.exog_names
        out["fixed_effects_posterior_mean"] = {n: float(v) for n, v in zip(names, f.fe_mean)}
        out["fixed_effects_posterior_sd"] = {n: float(v) for n, v in zip(names, f.fe_sd)}
        out["vc_log_sd_mean"] = {n: float(v) for n, v in zip(m.vcp_names, f.vcp_mean)}
    except Exception as e:  # noqa: BLE001
        out["error"] = str(e)[:200]
    tab = df.groupby(["site", "w"])["y"].mean().unstack().round(3)
    out["site_x_template_off_share_attempts"] = {s: {w: (None if pd.isna(v) else float(v)) for w, v in row.items()}
                                                 for s, row in tab.iterrows()}
    return out

In [ ]:
tr = transmission(rows)
pm = pooled_model(rows)
ref_tr = REF["transmission_disp_R35"]
dm = tr.get("disp_R35", {})
if "params" in dm:
    print(f"transmission GLM on {dm['n']} correct prefill-site programs (published: {ref_tr['n']}):")
    for k in ("is_S0", "logd"):
        print(f"  {k:6s} demo {dm['params'][k]:+.2f} [{dm['ci95'][k][0]:+.2f}, {dm['ci95'][k][1]:+.2f}]   "
              f"full {ref_tr['params'][k]:+.2f} [{ref_tr['ci95'][k][0]:+.2f}, {ref_tr['ci95'][k][1]:+.2f}]")
    print(f"  reading: demo '{dm.get('S0_reading')}' | full '{ref_tr['S0_reading']}'")
else:
    print("transmission GLM:", dm)
print("  converged:", {k: tr.get(k, {}).get("converged") for k in ("disp_R35", "disp_R35_sites", "first_token_kl")},
      "(published: disp_R35 True, disp_R35_sites False, first_token_kl True)")
print(f"first-token KL degenerate share (< 1e-6): {tr.get('ftkl_share_below_1e-6', float('nan')):.2f}")
print("\nmatched-dose quintile bins (off-default share among correct programs):")
for b in tr.get("matched_dose_bins", []):
    fmt = lambda v: "  n/a" if v is None else f"{v:5.2f}"
    print(f"  log d in [{b['logd_range'][0]:.2f}, {b['logd_range'][1]:.2f}]: S0 {fmt(b['off_S0'])} (n={b['n_S0']:3d})"
          f" | other sites {fmt(b['off_other'])} (n={b['n_other']:3d})")
print("\npooled mixed model, site effects vs S0 (posterior mean; demo | full):")
for k, v in (pm.get("fixed_effects_posterior_mean") or {}).items():
    if k.startswith("C(site") and ":" not in k or k == "la":
        print(f"  {k:36s} {v:+.2f} | {REF['pooled_model_fixed_effects_posterior_mean'][k]:+.2f}")
print(f"\nanalysis sections 5-11 took {time.time() - t_start:.0f}s")

## 12. Results: figures (`src/figures.py`)

These are the original figure functions. They read the in-memory results (`hull`, `cells`, `flips`, `rows`)
instead of `results/*.json`, and they show the figures inline instead of saving PDF and PNG files. All of them are
drawn from the **demo subset**, N = 15 per cell.

- **Fig 1.** The hull frontiers: correct-only Hill against pass-rate cost. A frontier rising high left of the 0.05
  line means the site buys diversity cheaply. A site whose correct programs never leave the default has a one-point
  frontier at Hill 1.0, which draws no line; that is why S1 and S2, and S5 on most tasks, do not appear. A negative
  cost means a mixture passes more often than the twin.
- **Fig 2.** The dose-response per site.
- **Fig 3.** The paired-flip matrix.
- **Fig 4.** Transmission: off-default share against downstream displacement.
- **Fig 7.** Phrasing clusters per idea, which measure surface variation.

In [ ]:
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
COL = {"S0": "#000000", "S1": "#E69F00", "S2": "#56B4E9", "S3": "#009E73", "S4": "#CC79A7", "S5": "#0072B2",
       "S1E": "#D55E00", "S2E": "#999999", "TOP5": "#D55E00", "TWIN": "#999999"}
SITE_NAME = {"S0": "S0 task spec, blocks 12-23", "S1": "S1 read position R", "S2": "S2 condition span",
             "S3": "S3 generated tokens 1-32", "S4": "S4 task spec, blocks 0-11", "S5": "S5 task spec, blocks 24-35"}


def save(fig, name: str) -> None:
    # notebook: show inline (the original wrote figures/<name>.pdf and .png)
    plt.show()


def fig1() -> None:
    h = hull["point"]
    fig, axs = plt.subplots(2, 3, figsize=(10, 5.6), sharey=True)
    for ax, t in zip(axs.ravel(), TASKS):
        for s in SITES:
            fr = h.get(f"{t}|{s}", {}).get("frontier")
            if fr:
                x, y = zip(*fr)
                ax.step(x, y, where="post", color=COL[s], lw=1.6 if s == "S0" else 1.1, label=s)
        ax.axvline(0.05, color="#bbbbbb", ls=":", lw=0.8)
        ax.set_title(t)
        ax.set_xlabel("pass-rate cost vs twin")
    axs[0, 0].set_ylabel("correct-only Hill-1 (depth 25)")
    axs[1, 0].set_ylabel("correct-only Hill-1 (depth 25)")
    axs[0, 0].legend(frameon=False, fontsize=7, ncol=2)
    fig.suptitle("Randomisation-hull frontiers per site (dotted: the H5 cost bound 0.05)")
    fig.tight_layout()
    save(fig, "fig1_hull_frontiers")


def fig2() -> None:
    c = cells
    fig, axs = plt.subplots(3, 6, figsize=(13, 6.2), sharex=True)
    for j, t in enumerate(TASKS):
        tw = c.get(f"{t}|TWIN|0", {})
        for s in SITES:
            xs, ex, pr, iv = [0], [0.0], [tw.get("pass_rate")], [tw.get("invalid_rate")]
            for a in (1, 2, 4):
                cc = c.get(f"{t}|{s}|{a}")
                if cc:
                    xs.append(a)
                    ex.append(cc.get("excess_correct_off") if cc.get("excess_correct_off") is not None else np.nan)
                    pr.append(cc.get("pass_rate"))
                    iv.append(cc.get("invalid_rate"))
            xx = [0.5 if x == 0 else x for x in xs]
            for i, v in enumerate((ex, pr, iv)):
                axs[i, j].plot(xx, v, marker="o", ms=3, color=COL[s], lw=1.5 if s == "S0" else 1, label=s)
        axs[0, j].set_title(t)
        for i in range(3):
            axs[i, j].set_xscale("log", base=2)
            axs[i, j].set_xticks([0.5, 1, 2, 4])
            axs[i, j].set_xticklabels(["twin", "1", "2", "4"])
    axs[0, 0].set_ylabel("correct-only off-default\nexcess over twin")
    axs[1, 0].set_ylabel("pass rate")
    axs[2, 0].set_ylabel("invalid rate")
    axs[0, 0].legend(frameon=False, fontsize=6)
    fig.supxlabel("alpha (per-position ban units)")
    fig.tight_layout()
    save(fig, "fig2_dose_response")


def fig3() -> None:
    pf = flips
    for a in (1, 2, 4):
        M = np.full((len(SITES) - 1, len(TASKS)), np.nan)
        P = np.full_like(M, np.nan)
        for i, s in enumerate(SITES[1:]):
            for j, t in enumerate(TASKS):
                r = pf.get(f"{t}|{s}|{a}")
                if r:
                    tb = r["table_attempts[both,S0only,siteonly,neither]"]
                    M[i, j] = tb[1] - tb[2]
                    P[i, j] = r["mcnemar_p_attempts"]
        if a == 2:
            fig, ax = plt.subplots(figsize=(6, 3.2))
            lim = np.nanmax(np.abs(M)) if np.isfinite(M).any() else 1
            im = ax.imshow(M, cmap="RdBu_r", vmin=-lim, vmax=lim)
            for i in range(M.shape[0]):
                for j in range(M.shape[1]):
                    if np.isfinite(M[i, j]):
                        ax.text(j, i, f"{int(M[i, j]):+d}\np={P[i, j]:.2g}", ha="center", va="center", fontsize=6)
            ax.set_xticks(range(len(TASKS)), TASKS)
            ax.set_yticks(range(len(SITES) - 1), SITES[1:])
            fig.colorbar(im, ax=ax, label="flips off default: S0-only minus site-only")
            # notebook: the slot count comes from the data (the original title said 30 shared slots)
            ax.set_title(f"Paired flips under CRN, alpha 2 ({pf['T03|S1|2']['n_pairs']} shared slots; exact McNemar p)")
            fig.tight_layout()
            save(fig, "fig3_paired_flips")


def fig4() -> None:
    rows_f = [r for r in rows if r.get("disp35") and r["correct"] and r["k"] < 6 and
              r["site"] in ("S0", "S1", "S2", "S4", "S5", "S1E", "S2E")]
    fig, ax = plt.subplots(figsize=(6, 4))
    for s in ("S0", "S1", "S2", "S4", "S5", "S1E", "S2E"):
        rs = [r for r in rows_f if r["site"] == s]
        if not rs:
            continue
        x = np.log10([r["disp35"] for r in rs])
        y = np.array([float(r["off"]) for r in rs])
        bins = np.quantile(x, np.linspace(0, 1, 5))
        mx, my = [], []
        for i in range(4):
            m = (x >= bins[i]) & (x <= bins[i + 1])
            if m.sum():
                mx.append(x[m].mean())
                my.append(y[m].mean())
        ax.plot(mx, my, marker="o", color=COL[s], label=s, lw=1.6 if s == "S0" else 1)
    ax.set_xlabel("log10 realised displacement at R, block 35 (vs clean twin)")
    ax.set_ylabel("off-default share among correct programs")
    ax.set_title("Transmission: off-default vs downstream dose (quartile bins per site)")
    ax.legend(frameon=False, fontsize=7)
    fig.tight_layout()
    save(fig, "fig4_transmission")


def fig7() -> None:
    c = cells
    fig, ax = plt.subplots(figsize=(8, 3.4))
    width = 0.12
    for i, s in enumerate(SITES):
        vals = []
        for a in (1, 2, 4):
            v = [c.get(f"{t}|{s}|{a}", {}).get("phrasing", {}).get("0.8", {}).get("clusters_per_10_programs")
                 for t in TASKS]
            v = [x for x in v if x is not None]
            vals.append(np.mean(v) if v else np.nan)
        ax.bar(np.arange(3) + (i - 2.5) * width, vals, width, color=COL[s], label=s)
    tw = [c.get(f"{t}|TWIN|0", {}).get("phrasing", {}).get("0.8", {}).get("clusters_per_10_programs") for t in TASKS]
    tw = [x for x in tw if x is not None]
    if tw:
        ax.axhline(np.mean(tw), color="#999999", ls="--", lw=1, label="twin")
    ax.set_xticks(range(3), ["alpha 1", "alpha 2", "alpha 4"])
    ax.set_ylabel("phrasing clusters per 10 correct\nprograms (theta 0.8, task mean)")
    ax.legend(frameon=False, fontsize=7, ncol=4)
    ax.set_title("Surface variation: AST-shingle phrasing clusters by site")
    fig.tight_layout()
    save(fig, "fig7_phrasing_clusters")

In [ ]:
for f in (fig1, fig2, fig3, fig4, fig7):
    try:
        f()
    except Exception as e:  # noqa: BLE001 - one broken figure must not stop the others
        print(f"{f.__name__} failed: {e}")

### Demo vs full run, and what the noise actually does to a program

**The comparison plot.** The left panel puts every (task, site) H5 from this demo subset against the published
full-run value. The S1 and S2 points sit under the S5 points at (1, 1). The right panel shows, per site, the share
of correct programs that left the task's default algorithm, pooled over the three doses.

**What the subset reproduces** (N = 15 per cell):

- **Inert sites.** S1 and S2 have H5 = 1.00 on every task, and none of their correct programs leaves the default
  (the full run: 0/526 and 0/527). S5 moves only T10.
- **Live sites.** Only the task-specification sites S0 and S4, and generated-token noise (S3), buy diversity.
- **Paired flips.** At alpha 2, S0 flips significantly more slots off the default than S1, S2 and S5 on T03 and
  T10.
- **Transmission.** At matched downstream displacement, the S0 indicator stays clearly above zero:
  site-specific, not generic transmission.

**What it does not reproduce.** The per-task ranking among the live sites. On this subset S0's H5 is lower than in
the full run on all six tasks, while S3 and S4 come out higher on several. So S0 has the best H5 only on T1 (full
run: T03, T1 and T2), and the frozen rule still says PARTIAL. That is the resolution limit the full run already
reported: N = 30 cannot rank two live sites within about 0.2-0.4 Hill, and every per-task margin was a bootstrap
close call. Halving N makes it worse.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
ax = axs[0]
for s in SITES:
    x = [REF["c1_site"]["per_task"][t]["H5"][s] for t in TASKS]
    y = [c1["per_task"][t]["H5"][s] for t in TASKS]
    ax.scatter(x, y, color=COL[s], label=s, s=36 if s == "S0" else 22, zorder=3)
lim = [0.9, max(3.0, max(max(v["H5"].values()) for v in c1["per_task"].values()) + 0.1)]
ax.plot(lim, lim, color="#bbbbbb", ls="--", lw=0.8)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel("H5, full run (N = 30 per cell)")
ax.set_ylabel(f"H5, demo subset (N = {len(DEMO_K) * len(TEMPLATE_IDS)} per cell)")
ax.set_title("Randomisation-hull price H5 per (task, site)")
ax.legend(frameon=False, fontsize=7, ncol=2)

ax = axs[1]
full_off = {"S0": None, "S1": 0, "S2": 0, "S5": 6}  # README: S1 0/526, S2 0/527, S5 6/521 (alpha 1-4, all tasks)
demo_off = {s: sum(1 for r in rows if r["site"] == s and r["correct"] and r["off"]) for s in SITES}
demo_cor = {s: sum(1 for r in rows if r["site"] == s and r["correct"]) for s in SITES}
ax.bar(range(len(SITES)), [demo_off[s] / max(demo_cor[s], 1) for s in SITES], color=[COL[s] for s in SITES])
for i, s in enumerate(SITES):
    ax.text(i, demo_off[s] / max(demo_cor[s], 1) + 0.005, f"{demo_off[s]}/{demo_cor[s]}", ha="center", fontsize=7)
short = {"S0": "S0\ntask spec\nblocks 12-23", "S1": "S1\nread pos. R", "S2": "S2\ncondition", "S3": "S3\ngen. tokens\n1-32",
         "S4": "S4\ntask spec\nblocks 0-11", "S5": "S5\ntask spec\nblocks 24-35"}
ax.set_xticks(range(len(SITES)), [short[s] for s in SITES], fontsize=7)
ax.set_ylabel("off-default share among correct programs")
ax.set_title("Correct programs that left the default algorithm\n(demo subset, alpha 1-4 pooled; "
             "full run: S1 0/526, S2 0/527, S5 6/521)")
fig.tight_layout()
plt.show()

In [ ]:
# One concrete flip: a slot whose twin writes the default algorithm and whose S0 arm writes a different correct one,
# while S1 (same direction, same dose units, read position instead of the task-spec states) stays on the default.
def show_flip(prefer=("T1", "T2", "T03", "T10", "T05", "T06")):
    idx = {(r["task"], r["w"], r["k"], r["arm"]): r for r in rows}
    for t in prefer:
        for a in ("S0_a1", "S0_a2", "S0_a4"):
            for w in TEMPLATE_IDS:
                for k in DEMO_K:
                    tw, r0, r1 = idx.get((t, w, k, "TWIN")), idx.get((t, w, k, a)), idx.get((t, w, k, a.replace("S0", "S1")))
                    if tw and r0 and r1 and tw["correct"] and not tw["off"] and r0["correct"] and r0["off"]:
                        print(f"slot {t}|{w}|{k}: twin = {tw['label']} | {a} = {r0['label']} | "
                              f"{a.replace('S0', 'S1')} = {r1['label']} (byte-identical to twin: "
                              f"{r1['code_sha256'] == tw['code_sha256']})")
                        for name, r in (("TWIN", tw), (a, r0)):
                            print(f"\n----- {name}: {r['label']} ({'pass' if r['passed'] else 'fail'}) -----")
                            print("\n".join(r["code"].splitlines()[:40]))
                        return
    print("no flip found in the selected slots")


show_flip()

## Summary

**What this demo reran.** On a 90-slot subset (N = 15 per cell), the notebook reran the artifact's pre-registered
analysis code end to end. It covered the hull prices with the bootstrap, the C1-site rule, the failure modes, the
CRN paired flips, the transmission GLM and the pooled model. It also checked the maths that injects the noise; the
CPU unit tests match the published values exactly.

**What the subset shows.** Read-out and condition noise (S1, S2) never changes the algorithm. Noise in the
task-specification states does, and does so beyond the displacement it transmits downstream. The ranking among the
live sites S0, S3 and S4 is too noisy at this N to settle.

**What the full run found** (N = 30 per cell, 2,000 bootstrap replicates):

- **Where the algorithm changes.** Only the task-specification states change the algorithm: S0 at blocks 12-23, and
  S4 at blocks 0-11, whose units are smaller.
- **Inert sites.** The read position (S1) and the condition sentence (S2) are inert, and so are the late blocks (S5)
  for all but one task: T10 shows weak movement.
- **Not generic transmission.** At matched downstream displacement, the S0 indicator stays at +2.15 logits.
- **The pre-registered verdict.** The C1-site rule is **PARTIAL**: S0 is best on 3 of 6 tasks, and every per-task
  margin is a close call.

**What it does not show.** That the S0 states encode a "plan". The experiment establishes a site-specific
perturbation effect in one model (Qwen3-8B, FP8) on six memorised tasks.

**Running the full scale.** Set `BOOT_B = 2000` and run on all 180 slots. The original pipeline's
`uv run method.py analysis --boot 2000 --workers 5` does this on the full `data/samples.jsonl`.